# imports

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import xarray as xr
import gcsfs

fs = gcsfs.GCSFileSystem()

# paths to directories of carbonate system calculation output

In [ ]:
# directory
phytoplankton_directory = 'gs://leap-persistent/{INSERT_USERNAME}/phytoplankton'

# paths to pyCO2sys calculation results
carbo_calc_paths = fs.glob(f'{phytoplankton_directory}/gridded_data_1850-2100/*/*/*/*.carbonate*.zarr')

# where to save merged datasets
result_save_dir = f'{phytoplankton_directory}/cmip6_wcarbonatecalcs_1850-2100'

# merge ESM output dataset with associated PyCO2SYS output dataset, and save

In [ ]:
for no, carbo_calc_path in enumerate(carbo_calc_paths):

    ### connecting carbonate system path with ESM/member path ###
    print(f'On path set number {no}')
    split_path_carbo = carbo_calc_path.split('/')
    scenario = split_path_carbo[4]
    model = split_path_carbo[5]
    member = split_path_carbo[6].split('_')[1]
    esm_output_path = f'{phytoplankton_directory}/gridded_data_1850-2100/{scenario}/{model}/member_{member}/{model}.{member}.Omon.zarr'

    esm_output = xr.open_zarr(esm_output_path)
    carbo_output = xr.open_zarr('gs://'+carbo_calc_path)

    ### merging CMIP6 output with carbonate system calculation for same ESM/member ###
    result = xr.merge([esm_output,carbo_output], compat="broadcast_equals")
    result_filename = f'/{scenario}/{model}/member_{member}/{model}.{member}.cmip6_wcarbonate_1850-2100.zarr'

    result = result.drop_encoding()
    ### saving ###
    result.to_zarr(result_save_dir + result_filename, zarr_format=2, mode='w')
    print(result_save_dir + result_filename)
    print('Saved!')